### 1. **移动平均法**

例 18.1 汽车配件某年 1～12 月份的化油器销售量 (单位：只) 统计数据见表 18.1 中第 2 行，试用一次移动平均法预测下一年 1 月份的销售量.  


### 表 18.1 化油器销售量及一次移动平均法预测值表  

| 月份 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 | 12 | 预测 |
| ---- | ---- | ---- | ---- | ---- | ---- | ---- | ---- | ---- | ---- | ---- | ---- | ---- | ---- |
| $y_i$ | 423 | 358 | 434 | 445 | 527 | 429 | 426 | 502 | 480 | 384 | 427 | 446 |  |
| $N = 3$ |  |  |  | 405 | 412 | 469 | 467 | 461 | 452 | 469 | 455 | 430 | 419 |
| $N = 5$ |  |  |  |  |  | 437 | 439 | 452 | 466 | 473 | 444 | 444 | 448 |  


分别取 $N = 3, N = 5$，按预测公式  
$$
\hat{y}_{t + 1}(3) = M_t^1(3) = \frac{y_t + y_{t - 1} + y_{t - 2}}{3}, \quad t = 3, 4, \cdots, 12,
$$  

$$
\hat{y}_{t + 1}(5) = M_t^1(5) = \frac{y_t + y_{t - 1} + y_{t - 2} + y_{t - 3} + y_{t - 4}}{5}, \quad t = 5, 6, \cdots, 12.
$$  

计算 3 个月和 5 个月移动平均预测值，分别见表 18.1 第 3 行和第 4 行. $N = 3$ 时，预测的标准误差为 $56.5752$；$N = 5$ 时，预测的标准误差为 $39.8159$.  


通过表 18.1 可以看到，实际数据波动较大，经移动平均后，随机波动明显减少，且 $N$ 越大，波动也越小. 同时，也可以看到，一次移动平均法的预测标准误差还是有些大，对于实际数据波动较大的序列，一般较少采用此法进行预测.  

In [ ]:
import numpy as np

y = np.array([423, 358, 434, 445, 527, 429, 426, 502, 480, 384, 427, 446])  # T=12

def MoveAverage(y, N):
    Mt = ['*'] * N
    # 窗口移动
    for i in range(N+1, len(y)+2):  # N+1~13
        M = y[i-(N+1):i-1].mean()  # (i-2)-(i-(N+1))=窗口大小N-1, 预测第N期
        Mt.append(round(M))
    return Mt

yt3 = MoveAverage(y, 3)  # 列表，size=13
s3 = np.sqrt(((y[3:] - yt3[3:-1]) ** 2).mean())  # y[3:](9,), yt3[3:-1]去掉最后一个预测值,mean() /(T - N)

yt5 = MoveAverage(y, 5)
s5 = np.sqrt(((y[5:] - yt5[5:-1]) ** 2).mean())

print(f"N=3时, 预测值: {yt3}, 预测的标准误差为: {s3}")
print(f"N=5时, 预测值: {yt5}, 预测的标准误差为: {s5}")

N=3时, 预测值: ['*', '*', '*', 405, 412, 469, 467, 461, 452, 469, 455, 430, 419], 预测的标准误差为: 56.60388679233962
N=5时, 预测值: ['*', '*', '*', '*', '*', 437, 439, 452, 466, 473, 444, 444, 448], 预测的标准误差为: 39.89808445097513


In [ ]:
# 简单移动平均使用的是等量加权策略，可以利用卷积convolve,滑动窗口（移动平均）

def sma(arr, n):
    weights = np.ones(n) / n
    return np.convolve(weights, arr)[n-1:-n+1]

In [ ]:
import numpy as np

y = np.array([423, 358, 434, 445, 527, 429, 426, 502, 480, 384, 427, 446])

n1 = 3
# 两个数，一个数也卷积
yt1 = np.convolve(np.ones(n1) / n1, y)[n1-1:-n1+1]  # 卷积结果(14,)len(a)+len(v)-1,a被卷积的数组，v卷积窗口，切片后[2:-2](10,)去掉索引为0，1，-1，-2的
s1 = np.sqrt(((y[n1:] - yt1[:-1]) ** 2).mean())

n2 = 5
yt2 = np.convolve(np.ones(n2) / n2, y)[n2-1:-n2+1]
s2 = np.sqrt(((y[n2:] - yt2[:-1]) ** 2).mean())

print(f"N=3时, 预测值: {yt1}, 预测的标准误差为: {s1}")
print(f"N=5时, 预测值: {yt2}, 预测的标准误差为: {s2}")

N=3时, 预测值: [405.         412.33333333 468.66666667 467.         460.66666667
 452.33333333 469.33333333 455.33333333 430.33333333 419.        ], 预测的标准误差为: 56.57519850976887
N=5时, 预测值: [437.4 438.6 452.2 465.8 472.8 444.2 443.8 447.8], 预测的标准误差为: 39.81586187868923


### 2. **指数平滑法**

#### 一次指数平滑法

例 18.2 某产品的 11 期价格如表 18.2 所示. 试预测该产品第 12 期的价格.  


### 表 18.2 某产品价格及指数平滑预测值计算表  

| 时期 $t$ | 价格 $y_t$ | 预测值 $\hat{y}_t$ ($\alpha = 0.2$) | 预测值 $\hat{y}_t$ ($\alpha = 0.5$) | 预测值 $\hat{y}_t$ ($\alpha = 0.8$) |
| ---- | ---- | ---- | ---- | ---- |
| 1 | 4.81 | 4.805 | 4.805 | 4.805 |
| 2 | 4.8 | 4.806 | 4.808 | 4.809 |
| 3 | 4.73 | 4.805 | 4.804 | 4.802 |
| 4 | 4.7 | 4.790 | 4.767 | 4.744 |
| 5 | 4.7 | 4.772 | 4.733 | 4.709 |
| 6 | 4.73 | 4.757 | 4.717 | 4.702 |
| 7 | 4.75 | 4.752 | 4.723 | 4.724 |
| 8 | 4.75 | 4.752 | 4.737 | 4.745 |
| 9 | 5.43 | 4.751 | 4.743 | 4.749 |
| 10 | 5.78 | 4.887 | 5.087 | 5.294 |
| 11 | 5.85 | 5.066 | 5.433 | 5.683 |
| 12 |  |  |  | 5.817 |  


采用指数平滑法, 并分别取 $\alpha = 0.2, 0.5$ 和 $0.8$ 进行计算, 初始值  
$$
S_0^{(1)} = \frac{y_1 + y_2}{2} = 4.805,
$$  
即  
$$
\hat{y}_1 = S_0^{(1)} = 4.805.
$$  

按预测模型  
$$
\hat{y}_{t + 1} = \alpha y_t + (1 - \alpha)\hat{y}_t,
$$  
计算各期预测值, 列于表 18.2 中.  


从表 18.2 可以看出, $\alpha = 0.2, 0.5$ 和 $0.8$ 时, 预测值是很不相同的. 究竟 $\alpha$ 取何值为好, 可通过计算它们的预测标准误差 $S$, 选取使 $S$ 较小的那个 $\alpha$ 值. 预测的标准误差见表 18.3. 计算结果表明 $\alpha = 0.8$ 时, $S$ 较小, 故选取 $\alpha = 0.8$, 该产品第 12 期价格的预测值为 $\hat{y}_{12} = 5.817$.  


### 表 18.3 预测的标准误差  

| $\alpha$ | $0.2$ | $0.5$ | $0.8$ |
| ---- | ---- | ---- | ---- |
| $S$ | $0.4148$ | $0.3216$ | $0.2588$ |  

In [6]:
import numpy as np
import pandas as pd
from regex import F

y = np.array([4.81, 4.8, 4.73, 4.7, 4.7, 4.73, 4.75, 4.75, 5.43, 5.78, 5.85])

def ExpMove(y, a):
    n = len(y)
    M = np.zeros(n)
    M[0] = (y[0] + y[1]) / 2
    for i in range(1, len(y)):
        M[i] = a * y[i-1] + (1-a) * M[i-1]
    return M

yt1 = ExpMove(y, 0.2)
yt2 = ExpMove(y, 0.5)
yt3 = ExpMove(y, 0.8)
s1 = np.sqrt(((y - yt1)**2).mean())
s2 = np.sqrt(((y - yt2)**2).mean())
s3 = np.sqrt(((y - yt3)**2).mean())

d = pd.DataFrame(np.c_[yt1, yt2, yt3])
with pd.ExcelWriter("一次指数平滑预测值.xlsx") as f:
    d.to_excel(f)
    
print(f"预测的标准误差分别为: {s1, s2, s3}")

yh = 0.8 * y[-1] + 0.2 * yt3[-1]
print(f"下一期的预测值为: {yh}")

预测的标准误差分别为: (0.4148362642161784, 0.32164247683489516, 0.25883473030674825)
下一期的预测值为: 5.8165517935616


#### 二次指数平滑法

例 18.3 已知某厂 10 期的钢产量如表 18.4 所示, 试预测第 11, 12 期的钢产量.  


### 表 18.4 某厂 10 期的钢产量及预测值  

| $t$ | 钢产量 $y_t$ | 一次平滑值 | 二次平滑值 | 预测值 $\hat{y}_t$ |
| ---- | ---- | ---- | ---- | ---- |
| 1 | 2031 | 2031 | 2031 | 2031 |
| 2 | 2234 | 2091.9 | 2049.27 | 2152.8 |
| 3 | 2566 | 2234.13 | 2104.728 | 2418.99 |
| 4 | 2820 | 2409.891 | 2196.277 | 2715.054 |
| 5 | 3006 | 2588.724 | 2314.011 | 2981.171 |
| 6 | 3093 | 2740.007 | 2441.81 | 3166.002 |
| 7 | 3277 | 2901.105 | 2579.598 | 3360.4 |
| 8 | 3514 | 3084.973 | 2731.211 | 3590.348 |
| 9 | 3770 | 3290.481 | 2898.992 | 3849.752 |
| 10 | 4107 | 3535.437 | 3089.925 | 4171.882 |
| 11 |  |  |  | 4362.815 |
| 12 |  |  |  |  |  


取 $\alpha = 0.3$, 初始值 $S_0^{(1)}$ 和 $S_0^{(2)}$ 都取序列的首项数值, 即 $S_0^{(1)} = S_0^{(2)} = 2031$. 计算 $S_t^{(1)}, S_t^{(2)}$, 列于表 18.4, 得到  
$$
S_{10}^{(1)} = 3535.437, \quad S_{10}^{(2)} = 3089.925.
$$  

由公式 (18.11), 可得 $t = 10$ 时  
$$
\begin{align*}
a_{10} &= 2S_{10}^{(1)} - S_{10}^{(2)} = 3980.9484, \\
b_{10} &= \frac{\alpha}{1 - \alpha}(S_{10}^{(1)} - S_{10}^{(2)}) = 190.9335,
\end{align*}
$$  
于是, 得 $t = 10$ 时直线趋势方程为  
$$
\hat{y}_{10 + m} = 3980.9484 + 190.9335m.
$$  


预测第 11, 12 期的钢产量为  
$$
\hat{y}_{11} = \hat{y}_{10 + 1} = 4171.8819, \quad \hat{y}_{12} = \hat{y}_{10 + 2} = 4362.8154.
$$  


利用  
$$
\hat{y}_{t + 1} = 2S_t^{(1)} - S_t^{(2)} + \frac{\alpha}{1 - \alpha}(S_t^{(1)} - S_t^{(2)}), \quad t = 0, 1, \cdots, 9,
$$  
求已知各期的预测值. 计算结果见表 18.4. 

In [ ]:
import numpy as np
import pandas as pd

y = np.loadtxt("Pdata18_3.txt")
n = len(y)  # 10
alpha = 0.3
yh = np.zeros(n)
s1 = np.zeros(n)
s2 = np.zeros(n)
s1[0] = y[0]
s2[0] = y[0]

for i in range(1, n):  # 1~9
    s1[i] = alpha * y[i] + (1 - alpha) * s1[i-1]  # 一次指数平滑法
    s2[i] = alpha * s1[i] + (1 - alpha) * s2[i-1]  # 二次指数平滑法
    yh[i] = 2 * s1[i-1] - s2[i-1] + alpha / (1-alpha) * (s1[i-1] - s2[i-1])  # 直线趋势模型
at = 2 * s1[-1] - s2[-1]  # a_t
bt = alpha / (1-alpha) * (s1[-1] - s2[-1])  # b_t
m = np.array([1, 2])  # 预测两次
yh2 = at + bt * m
print(f"预测值为: {yh2}")

y = np.array(y)
yh = np.delete(yh, 0)
yh = np.append(yh, yh2)
y = np.append(y, 0)
s1 = np.append(s1, 0)
s2 = np.append(s2, 0)
d = pd.DataFrame(np.c_[y, s1, s2, yh])
with pd.ExcelWriter ("二次指数平滑预测值.xlsx", engine='openpyxl') as f:
    d.to_excel(f, sheet_name='钢产量', index=False)
df = pd.read_excel("二次指数平滑预测值.xlsx")
df.columns = ['钢产量', '一次平滑值', '二次平滑值', '预测值']
df

预测值为: [4171.88192538 4362.81543832]


,钢产量,一次平滑值,二次平滑值,预测值
0,2031,2031.000000,2031.000000,2031.000000
1,2234,2091.900000,2049.270000,2152.800000
2,2566,2234.130000,2104.728000,2418.990000
3,2820,2409.891000,2196.276900,2715.054000
4,3006,2588.723700,2314.010940,2981.170500
5,3093,2740.006590,2441.809635,3166.002240
6,3277,2901.104613,2579.598128,3360.399591
7,3514,3084.973229,2731.210659,3590.348330
8,3770,3290.481260,2898.991839,3849.751862
9,4107,3535.436882,3089.925352,4171.881925


### 3. 季节模型

例 18.4 某商店按季度统计的 3 年 (12 个季度) 冰箱的销售数据 (单位：万元) 见表 18.5. 求 2004 年 4 个季度的销售额.  


### 表 18.5 某商店 12 个季度冰箱销售资料  

| 年份 | 一季度 | 二季度 | 三季度 | 四季度 |
| ---- | ---- | ---- | ---- | ---- |
| 2001 | 265 | 373 | 333 | 266 |
| 2002 | 251 | 379 | 374 | 309 |
| 2003 | 272 | 437 | 396 | 348 |  


把表 18.5 中 3 行 4 列总共 12 个数据保存到文本文件 `Pdata18_4.txt` 中, 利用 Python 软件, 求得 2004 年 4 个季度的销售额分别为 $269.7534$ 万元、$407.0263$ 万元、$377.5862$ 万元、$315.9674$ 万元.  

In [ ]:
import numpy as np

a = np.loadtxt("Pdata18_4.txt")
m, n = a.shape
amean = a.mean()
cmean = a.mean(axis=0)  # 第0个维度代表行，沿着行的方向，逐列求平均,所有年季度平均
r = cmean / amean
w = np.arange(1, m+1)  
yh = w.dot(a.sum(axis=1)) / w.sum()  # 计算下一年的预测值
yj = yh / n  # 计算预测各月份的季度平均值
yjh = yj * r  # 计算季度的预测值
print(f"下一年个季度的预测值为: {yjh}")

下一年个季度的预测值为: [269.75335165 407.0263136  377.586227   315.96744109]
